In [1]:
import numpy as np
np.random.seed(42)

In [21]:
class LinearModel:
    def __init__(self, n_features: int) -> None:
        self.W = np.random.randn(n_features, 1)
        self.b = np.zeros((1, 1))

    def sigmoid(self, x: np.ndarray) -> np.ndarray:
        return 1 / (1 + np.exp(-x))

    def forward(self, x: np.ndarray) -> np.ndarray:
        z = x @ self.W + self.b  # (10, 5) @ (5, 1) + (1, 1) ==> (10, 1)
        return self.sigmoid(z)

    def backward(self, x: np.ndarray, y: np.ndarray, p: np.ndarray):
        n = x.shape[0]
        dz = (p - y) / n                         # (10, 1) sigmoid + BCE cancel to this
        dW = x.T @ dz                            # (5, 10) @ (10, 1) ==> (5, 1) same shape as W
        db = np.sum(dz, axis=0, keepdims=True)   # (1, 1)
        return dW, db

    def step(self, dW: np.ndarray, db: np.ndarray, lr: float) -> None:
        self.W = self.W - lr * dW
        self.b = self.b - lr * db


np.random.seed(42)
x_train = np.random.randn(10, 5)
y_train = np.array([0, 0, 0, 1, 1, 1, 0, 0, 1, 1]).reshape(-1, 1)

model = LinearModel(n_features=x_train.shape[1])
lr = 1.0
epochs = 2000

for epoch in range(epochs):
    p = model.forward(x_train)
    loss = np.mean(-1 * (y_train * np.log(p) + (1 - y_train) * np.log(1 - p)))
    dW, db = model.backward(x_train, y_train, p)
    model.step(dW, db, lr)
    if epoch % 200 == 0:
        print(f"epoch {epoch:4d} | loss {loss:.6f}")

print(f"epoch {epochs:4d} | loss {loss:.6f}")


epoch    0 | loss 1.102216
epoch  200 | loss 0.250506
epoch  400 | loss 0.191870
epoch  600 | loss 0.153277
epoch  800 | loss 0.126423
epoch 1000 | loss 0.106936
epoch 1200 | loss 0.092292
epoch 1400 | loss 0.080961
epoch 1600 | loss 0.071975
epoch 1800 | loss 0.064697
epoch 2000 | loss 0.058725


In [63]:
x_train = np.random.randn(1000, 2)
xq = np.random.randn(2, 2)
np.random.shuffle(x_train)
y_train = np.repeat([0,1], [500, 500])
np.random.shuffle(y_train)


In [ ]:
class KNNClassifier:
    def __init__(self, k, weights: str) -> None:
        self.k = k
        self.weights = weights

    def fit(self, x:np.ndarray, y:np.ndarray):
        self.x = x
        self.y = y
        self.k = min(self.k, x.shape[0])
        self.num_features = x.shape[1]
        self.classes = np.unique(y)
        return self

    def _neighbours(self, xq: np.ndarray)->tuple[np.ndarray, np.ndarray]:
        assert self.num_features == xq.shape[1], f"number of features must be same"
        # a2 = np.einsum("ij","ij->i", xq, xq)[:, None]
        # b2 = np.einsum("ij", "ij->i" , self.x, self.x) [None, :]
        a2 = np.einsum("ij,ij->i", xq, xq)[:, None]
        b2 = np.einsum("ij,ij->i", self.x, self.x)[None, :]

        distance_matrix = (a2+b2) - 2*(xq@self.x.T)

        idx = np.argpartition(distance_matrix, self.k-1)[...,:self.k]
        d = np.take_along_axis(distance_matrix, idx)
        order = np.argsort(d, axis=-1)
        return np.take_along_axis(idx, order, axis=1), np.take_along_axis(d, order, axis=1)

    def predict_proba(self, xq: np.ndarray):
        idxs, distances = self._neighbours(xq)
        pred_lbl = self.y[idxs] # shape is (number of queries, k)

        if self.weights == "distance":
            w = 1 / (np.sqrt(distances))

        C = len(self.classes)
        votes = np.zeros((len(xq), C))

        for ci, c in enumerate(self.classes):
            votes[:, ci] = (w*(pred_lbl==c)).sum(axis=-1)

        prob_pred = votes / np.sum(votes, axis=1, keepdims=True)
        
        return prob_pred


In [128]:
k = KNNClassifier(k=2, weights="distance")

k.fit(x_train, y_train)
k.predict_proba(xq)

array([[1., 0.],
       [0., 1.]])

In [48]:
y_train

array([1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0,
       1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1,
       1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0])